# Train Phishing URL Detection Model

This notebook trains an XGBoost classifier using extracted URL features, evaluates it, and saves the model and feature importance.

## 1) Imports

In [ ]:
import os
import numpy as np
import pandas as pd
import xgboost as xgb

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from sklearn.model_selection import train_test_split

## 2) Configuration

In [ ]:
from pathlib import Path

ARTIFACTS_DIR = Path(os.getenv('MODEL_ARTIFACTS_DIR', '.')).resolve()

CONFIG = {
    'data_file': os.getenv('MODEL_DATASET_PATH', str(ARTIFACTS_DIR / 'fast_url_features.csv')),
    'data_file_fallbacks': [str(ARTIFACTS_DIR / 'fast_url_html_features.csv')],
    'model_file': os.getenv('MODEL_PATH', str(ARTIFACTS_DIR / 'xgboost_phishing_model.json')),
    'feature_importance_file': str(ARTIFACTS_DIR / 'feature_importance.csv'),
    'test_size': 0.2,
    'validation_size': 0.15,
    'random_state': 42,
    'model_params': {
        'n_estimators': 2500,
        'max_depth': 6,
        'tree_method': 'hist',
        'scale_pos_weight': 1,
        'learning_rate': 0.1,
        'min_child_weight': 1,
        'subsample': 0.8,
        'colsample_bytree': 0.8,
        'gamma': 0,
        'reg_alpha': 0.1,
        'reg_lambda': 1.0,
        'random_state': 42,
        'n_jobs': -1,
        'eval_metric': 'logloss'
    }
}

print(f"Using artifacts dir: {ARTIFACTS_DIR}")
CONFIG

## 3) Helper Functions

In [ ]:
def resolve_data_file(primary_file, fallback_files):
    candidates = [primary_file] + fallback_files
    for candidate in candidates:
        if os.path.exists(candidate):
            if candidate != primary_file:
                print(f"Primary feature file not found. Using fallback: {candidate}")
            return candidate
    raise FileNotFoundError(f"None of the configured feature files exist: {candidates}")


def load_data(file_path):
    print("Loading feature dataset...")
    data = pd.read_csv(file_path, low_memory=False)
    print(f"Dataset shape: {data.shape}")
    print("\nClass distribution:")
    print(data['Label'].value_counts())
    return data


def preprocess_features(X):
    print("\nFeature data types:")
    print(X.dtypes.value_counts())

    bool_cols = X.select_dtypes(include=['bool']).columns
    if len(bool_cols) > 0:
        print(f"Converting {len(bool_cols)} boolean columns to integers")
        X[bool_cols] = X[bool_cols].astype(int)

    object_cols = X.select_dtypes(include=['object', 'string']).columns
    if len(object_cols) > 0:
        print(f"Found {len(object_cols)} object/string columns: {object_cols.tolist()}")
        for col in object_cols:
            unique_vals = X[col].dropna().unique()
            print(f"  {col}: unique values = {unique_vals[:5]}")
            if set(unique_vals).issubset({'True', 'False', True, False}):
                X[col] = X[col].map({'True': 1, 'False': 0, True: 1, False: 0})
                print(f"  -> Converted {col} from boolean strings to int")
            else:
                print(f"  -> Dropping {col} (cannot convert)")
                X = X.drop(col, axis=1)

    if X.isnull().any().any():
        missing_count = X.isnull().sum().sum()
        print(f"\nFilling {missing_count} missing values with 0")
        X = X.fillna(0)

    return X


def split_data(X, y, test_size, validation_size, random_state):
    print("\nSplitting data (train/val/test)...")
    X_temp, X_test, y_temp, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state, stratify=y
    )

    val_ratio = validation_size / (1 - test_size)
    X_train, X_val, y_train, y_val = train_test_split(
        X_temp, y_temp, test_size=val_ratio, random_state=random_state, stratify=y_temp
    )

    print(f"Training set: {X_train.shape}")
    print(f"Validation set: {X_val.shape}")
    print(f"Test set: {X_test.shape}")
    return X_train, X_val, X_test, y_train, y_val, y_test


def train_model(X_train, y_train, X_val, y_val, model_params):
    print("\nTraining XGBoost model with early stopping...")
    params = model_params.copy()
    params['early_stopping_rounds'] = 15

    model = xgb.XGBClassifier(**params)
    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        verbose=False
    )

    print(f"Best iteration: {model.best_iteration}")
    print(f"Best validation score: {model.best_score:.4f}")
    return model


def evaluate_model(model, X_test, y_test):
    print("\nMaking predictions...")
    y_pred = model.predict(X_test)

    print("\n" + "=" * 50)
    print("MODEL EVALUATION")
    print("=" * 50)

    accuracy = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    print(f"\nAccuracy: {accuracy:.4f}")
    print(f"F1 Score: {f1:.4f}")

    cm = confusion_matrix(y_test, y_pred)
    print("\nConfusion Matrix:")
    print(cm)
    print(f"  True Negatives:  {cm[0][0]:,}")
    print(f"  False Positives: {cm[0][1]:,}")
    print(f"  False Negatives: {cm[1][0]:,}")
    print(f"  True Positives:  {cm[1][1]:,}")

    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))
    return y_pred


def save_model_and_features(model, feature_names, model_file, importance_file):
    model.save_model(model_file)
    print(f"\nModel saved to: {model_file}")

    feature_importance = pd.DataFrame({
        'feature': feature_names,
        'importance': model.feature_importances_
    }).sort_values('importance', ascending=False)

    print("\nTop 20 Most Important Features:")
    print(feature_importance.head(20))

    feature_importance.to_csv(importance_file, index=False)
    print(f"Feature importance saved to: {importance_file}")
    return feature_importance

## 4) Run Training Pipeline

In [ ]:
# Load
data_file = resolve_data_file(CONFIG['data_file'], CONFIG['data_file_fallbacks'])
data = load_data(data_file)

# Prepare features/labels
X = data.drop('Label', axis=1)
y = data['Label']

# Preprocess
X = preprocess_features(X)

# Split
X_train, X_val, X_test, y_train, y_val, y_test = split_data(
    X, y,
    CONFIG['test_size'],
    CONFIG['validation_size'],
    CONFIG['random_state']
)

# Train
model = train_model(X_train, y_train, X_val, y_val, CONFIG['model_params'])

# Evaluate
_ = evaluate_model(model, X_test, y_test)

# Save artifacts
feature_importance = save_model_and_features(
    model,
    X.columns,
    CONFIG['model_file'],
    CONFIG['feature_importance_file']
)

print("\nTraining pipeline completed successfully!")

## 5) View Top Features

In [ ]:
feature_importance.head(20)